# 🌊 Molecular Surfaces

---

## Learning Objectives

- Understand different molecular surface representations
- Calculate solvent-accessible surface area (SASA)
- Visualize hydrophobic and electrostatic surfaces
- Find binding pockets and cavities

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This notebook is designed to bridge concept to practical analysis workflows.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Inspect assumptions before running model/statistical code.
- Track input/output files for reproducibility.
- Interpret outputs with biological context, not numbers alone.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

## Types of Molecular Surfaces

```
┌─────────────────────────────────────────────────────────────────────────┐
│                      MOLECULAR SURFACE TYPES                            │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  1. VAN DER WAALS SURFACE                                               │
│     ○───○                                                               │
│    ╱     ╲    Simply the VdW radii of atoms                            │
│   ○   ○   ○   Shows atomic size                                        │
│    ╲     ╱    Has gaps between atoms                                   │
│     ○───○                                                               │
│                                                                         │
│  2. SOLVENT-ACCESSIBLE SURFACE (SAS)                                    │
│        ╭─────╮                                                          │
│       ╱  ···  ╲   Traced by center of probe sphere (1.4 Å)            │
│      │  ○───○  │  Rolling probe around molecule                        │
│       ╲  ···  ╱   Larger than VdW surface                              │
│        ╰─────╯                                                          │
│                                                                         │
│  3. SOLVENT-EXCLUDED SURFACE (SES) / Molecular Surface                  │
│        ╭────╮                                                           │
│       ╱      ╲    Contact + reentrant surface                          │
│      │  ○──○  │   Where solvent cannot reach                           │
│       ╲      ╱    Smooth, continuous                                   │
│        ╰────╯     Most realistic for visualization                     │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

---

## Solvent-Accessible Surface Area (SASA)

SASA measures how much surface area is accessible to solvent molecules.

```
    Probe sphere (water, r=1.4 Å)
           ○
          ╱ ╲
    ─────○   ○─────  SAS (dashed line)
        ╱     ╲
       ○───────○     Atom surface
       
    SAS = path traced by probe center
    
    Typical SASA values:
    ┌──────────────┬────────────────────┐
    │ Residue      │ Max SASA (Å²)      │
    ├──────────────┼────────────────────┤
    │ ALA          │ 113                │
    │ ARG          │ 241                │
    │ TRP          │ 259                │
    │ GLY          │ 85                 │
    └──────────────┴────────────────────┘
```

In [ ]:
import numpy as np

# Van der Waals radii (Å)
VDW_RADII = {
    'C': 1.70,
    'N': 1.55,
    'O': 1.52,
    'S': 1.80,
    'H': 1.20,
    'P': 1.80,
    'FE': 2.00,
    'ZN': 1.39,
    'CA': 1.97,
    'MG': 1.73
}

# Maximum SASA for amino acids (fully exposed)
MAX_SASA = {
    'ALA': 113, 'ARG': 241, 'ASN': 158, 'ASP': 151,
    'CYS': 140, 'GLN': 189, 'GLU': 183, 'GLY': 85,
    'HIS': 194, 'ILE': 182, 'LEU': 180, 'LYS': 211,
    'MET': 204, 'PHE': 218, 'PRO': 143, 'SER': 122,
    'THR': 146, 'TRP': 259, 'TYR': 229, 'VAL': 160
}

def calculate_sasa_simple(atoms, probe_radius=1.4, n_points=100):
    """
    Simple SASA calculation using Shrake-Rupley algorithm.
    
    Parameters:
        atoms: List of dicts with 'x', 'y', 'z', 'element'
        probe_radius: Solvent probe radius (Å)
        n_points: Number of test points per atom
        
    Returns:
        Total SASA in Å²
    """
    # Generate uniform points on sphere
    golden_angle = np.pi * (3 - np.sqrt(5))
    indices = np.arange(n_points)
    y = 1 - (indices / (n_points - 1)) * 2
    radius = np.sqrt(1 - y * y)
    theta = golden_angle * indices
    
    sphere_points = np.column_stack([
        radius * np.cos(theta),
        y,
        radius * np.sin(theta)
    ])
    
    total_sasa = 0
    
    for i, atom in enumerate(atoms):
        element = atom.get('element', 'C').upper()
        vdw_r = VDW_RADII.get(element, 1.70)
        r = vdw_r + probe_radius
        
        # Points on expanded sphere around atom
        points = sphere_points * r + np.array([atom['x'], atom['y'], atom['z']])
        
        # Check which points are buried by other atoms
        n_accessible = 0
        for point in points:
            accessible = True
            for j, other in enumerate(atoms):
                if i == j:
                    continue
                other_element = other.get('element', 'C').upper()
                other_r = VDW_RADII.get(other_element, 1.70) + probe_radius
                
                dist = np.sqrt(
                    (point[0] - other['x'])**2 +
                    (point[1] - other['y'])**2 +
                    (point[2] - other['z'])**2
                )
                if dist < other_r:
                    accessible = False
                    break
            if accessible:
                n_accessible += 1
        
        # Surface area contribution
        atom_sasa = 4 * np.pi * r**2 * (n_accessible / n_points)
        total_sasa += atom_sasa
    
    return total_sasa

# Example with 3 atoms
test_atoms = [
    {'x': 0, 'y': 0, 'z': 0, 'element': 'C'},
    {'x': 1.5, 'y': 0, 'z': 0, 'element': 'C'},
    {'x': 0.75, 'y': 1.3, 'z': 0, 'element': 'N'}
]

sasa = calculate_sasa_simple(test_atoms, n_points=50)
print(f"Total SASA for 3-atom system: {sasa:.1f} Å²")

---

## Surface Properties

### Hydrophobicity Surface

Color by hydrophobicity to identify:
- Hydrophobic patches (protein-protein interfaces)
- Membrane-interacting regions
- Ligand binding sites

### Electrostatic Surface

Color by charge to identify:
- Positively charged regions (blue)
- Negatively charged regions (red)
- DNA/RNA binding sites
- Active site residues

In [ ]:
# Kyte-Doolittle hydrophobicity scale
HYDROPHOBICITY = {
    'ALA': 1.8,  'ARG': -4.5, 'ASN': -3.5, 'ASP': -3.5,
    'CYS': 2.5,  'GLN': -3.5, 'GLU': -3.5, 'GLY': -0.4,
    'HIS': -3.2, 'ILE': 4.5,  'LEU': 3.8,  'LYS': -3.9,
    'MET': 1.9,  'PHE': 2.8,  'PRO': -1.6, 'SER': -0.8,
    'THR': -0.7, 'TRP': -0.9, 'TYR': -1.3, 'VAL': 4.2
}

def classify_residue_exposure(residue_sasa, residue_name):
    """
    Classify residue as buried/intermediate/exposed.
    
    Parameters:
        residue_sasa: Calculated SASA for this residue
        residue_name: 3-letter amino acid code
        
    Returns:
        Classification string
    """
    max_sasa = MAX_SASA.get(residue_name, 150)
    relative_sasa = residue_sasa / max_sasa
    
    if relative_sasa < 0.25:
        return 'buried'
    elif relative_sasa < 0.50:
        return 'intermediate'
    else:
        return 'exposed'

# Example
print("Residue exposure classification:")
print("  ALA with 20 Å² SASA:", classify_residue_exposure(20, 'ALA'))
print("  ALA with 60 Å² SASA:", classify_residue_exposure(60, 'ALA'))
print("  ALA with 100 Å² SASA:", classify_residue_exposure(100, 'ALA'))

---

## Jmol Surface Commands

```javascript
// Load structure
load =1crn

// Solvent-accessible surface
isosurface sasurface molecular

// Color by hydrophobicity
isosurface sasurface colorscheme "rwb" map property partialcharge

// Transparent surface with cartoon inside
isosurface sasurface translucent 0.5
cartoon on

// Calculate electrostatic potential (requires APBS)
isosurface esp colorscheme "bwr" map electrostatic

// Find cavities
isosurface pocket cavity
```

---

## 🏋️ Exercises

### Exercise 1: Buried vs Exposed
Calculate SASA for each residue and classify as buried/exposed.

### Exercise 2: Hydrophobic Clusters
Identify clusters of hydrophobic residues on the protein surface.

### Exercise 3: Interface Analysis
Calculate the buried surface area at a protein-protein interface.

---

## 📚 Resources

- [FreeSASA](https://freesasa.github.io/)
- [NACCESS](http://www.bioinf.manchester.ac.uk/naccess/)
- [CASTp Server](http://sts.bioe.uic.edu/castp/) - Pocket detection

---
**Navigation:** [Back to Course README](../../README.md) · [Open this notebook path](`Course/Archive/07_Kodomo_Structural_Bioinformatics/06_Molecular_Surfaces/01_Surface_Analysis.ipynb`)
